# Week 2 NumPy Assignment 



In [3]:
import numpy as np
np.random.seed(42)

## Part A — Array Manipulation

### Problem 1 — Temperature Converter
 use `np.multiply` and `np.add` explicitly instead of operators. Functionally identical, but shows the underlying ufuncs operator syntax calls under the hood.

In [4]:
def celsius_to_fahrenheit(celsius_array):
    return np.add(np.multiply(celsius_array, 9/5), 32)

test_c = np.array([0, 20, 37, 100])
print(celsius_to_fahrenheit(test_c))  # expected: [ 32.  68.  98.6 212.]

[ 32.   68.   98.6 212. ]


### Problem 2 — Filter High-Value Transactions
 `np.where` returns indices where the condition holds; fancy-index with those instead of a raw boolean mask.

In [5]:
def filter_high_value(transactions, threshold=50000):
    idx = np.where(transactions >= threshold)
    return transactions[idx]

test_tx = np.array([12000, 55000, 49999, 100000, 50000])
print(filter_high_value(test_tx))  # expected: [ 55000 100000  50000]

[ 55000 100000  50000]


### Problem 3 — Reshape Sensor Data
 use `-1` for one dimension so NumPy infers it, instead of hardcoding both dims.

In [6]:
def reshape_to_days(readings):
    return readings.reshape(-1, 24)  # NumPy infers 7 from the array length

test_readings = np.arange(168)
result = reshape_to_days(test_readings)
print("shape:", result.shape)
print("day 1:", result[0])

shape: (7, 24)
day 1: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


### Problem 4 — Daily Min/Max Temperature
use `np.amin`/`np.amax` (the explicit function form) instead of the `.min()`/`.max()` array methods. Same result, different calling style.

In [7]:
def daily_min_max(daily_readings):
    return np.amin(daily_readings, axis=1), np.amax(daily_readings, axis=1)

test_data = reshape_to_days(np.arange(168))
mins, maxs = daily_min_max(test_data)
print("daily mins:", mins)
print("daily maxs:", maxs)

daily mins: [  0  24  48  72  96 120 144]
daily maxs: [ 23  47  71  95 119 143 167]


## Part B — Matrix Operations

### Problem 5 — Total Revenue by Product
**Alternative:** use `np.dot` with a ones-vector to sum over days, instead of `.sum(axis=0)` — reframes the row-sum as a matrix product.

In [8]:
def total_revenue_by_product(units_sold, unit_price):
    n_days = units_sold.shape[0]
    ones = np.ones(n_days)
    total_units_per_product = np.dot(ones, units_sold)  # (n_days,) @ (n_days,n_products) -> (n_products,)
    return total_units_per_product * unit_price

test_units = np.array([[10, 5], [8, 6], [12, 4]])
test_price = np.array([1000, 2000])
print(total_revenue_by_product(test_units, test_price))  # expected: [30000 30000]

[30000. 30000.]


### Problem 6 — Grade Book Weighted Average
use `np.dot` explicitly instead of the `@` operator — they do the same thing, `@` is just shorthand for `np.dot`/`np.matmul` on these shapes.

In [9]:
def weighted_final_scores(scores, weights):
    return np.dot(scores, weights)

test_scores = np.array([[90, 80, 70], [60, 70, 80], [100, 90, 95]])
test_weights = np.array([0.5, 0.3, 0.2])
print(weighted_final_scores(test_scores, test_weights))  # expected: [83. 67. 96.]

[83. 67. 96.]


### Problem 7 — Solve for Ingredient Cost
`np.linalg.solve` on the coefficient matrix. Included for completeness.

In [10]:
def solve_ingredient_costs():
    A = np.array([[2, 1],
                  [1, 3]])
    b = np.array([3000, 4000])
    return np.linalg.solve(A, b)

print(solve_ingredient_costs())  # expected: [1000. 1000.]

[1000. 1000.]


## Part C — Simple Simulations

### Problem 8 — Coin Flip Simulation
 use `np.random.choice([0, 1], size=...)` instead of `randint` — a different sampling function producing the same kind of result.

In [11]:
def simulate_coin_flips(n_flips=50000, seed=42):
    np.random.seed(seed)
    flips = np.random.choice([0, 1], size=n_flips)
    return flips.mean()

print(simulate_coin_flips())  # expected: a value close to 0.5

0.49908


### Problem 9 — Customer Wait Time Simulation
use `np.count_nonzero` on the boolean mask divided by length, instead of `.mean() * 100` — same math, spelled out differently.

In [12]:
def pct_waiting_over_7min(n_customers=10000, seed=42):
    np.random.seed(seed)
    wait_times = np.random.normal(5, 1.5, size=n_customers)
    over_7 = wait_times > 7
    return np.count_nonzero(over_7) / len(over_7) * 100

print(pct_waiting_over_7min())  # expected: roughly 9-10 (percent)

9.229999999999999


### Problem 10 — Simple Portfolio Growth Simulation
 use `np.prod` directly on the growth factors (since you only need the *final* value, not every intermediate day) instead of `np.cumprod(...)[-1]`. More efficient when you don't need the full path.

In [13]:
def simulate_portfolio_growth(start_value=1_000_000, n_days=252, seed=42):
    np.random.seed(seed)
    daily_returns = np.random.normal(0.0005, 0.015, size=n_days)
    growth_factors = 1 + daily_returns
    return start_value * np.prod(growth_factors)

print(simulate_portfolio_growth())

1089172.0041520006


## Stretch Challenge (Optional)
 use `np.prod(axis=1)` for the final prices directly (since only the endpoint is needed) instead of building the full `cumprod` price path.

In [14]:
def monte_carlo_portfolio(n_simulations=1000, start_value=1_000_000, n_days=252, seed=42):
    np.random.seed(seed)
    daily_returns = np.random.normal(0.0005, 0.015, size=(n_simulations, n_days))
    growth_factors = 1 + daily_returns
    final_prices = start_value * np.prod(growth_factors, axis=1)
    mean_final_price = final_prices.mean()
    pct_below_start = (final_prices < start_value).mean() * 100
    return mean_final_price, pct_below_start

print(monte_carlo_portfolio())

(np.float64(1133520.334959366), np.float64(33.4))
